# Diseño óptimo de perfiles por grupo (AISC 360-16 ASD + NCh2369:2025)

Sube a Kaggle **solo el modelo SAP2000 con resultados** (`.s2k` o `.txt`). Todo lo demás está en el notebook.

## Contenido
1. **Configuración general.** Materiales (Fy, Ry, E), factor de utilización por defecto y rutas.
2. **Lectura del modelo.** Lee el archivo SAP2000: unidades, Ω, conectividad, grupos, secciones y fuerzas ASD.
3. **Ingreso de parámetros por grupo.** Formulario para cada grupo que detecta el modelo: tipo, familias, Kx, Ky, Lb, Cb, UC máximo, límite KL/r y excepción 8.8.4.
4. **Barras, grupos y demandas.** Une barras con grupos y arma las demandas por combinación y estación.
5. **Base de datos de perfiles (oculta).** 600 perfiles validados, codificados. No requiere edición.
6. **Catálogo de propiedades.** Convierte la base a unidades de cálculo (mm, N).
7. **Verificación ASD y límites de esbeltez.** Compresión, flexión con pandeo lateral-torsional, corte, interacción H1, esbeltez local (NCh2369 Tabla 9) y global (KL/r).
8. **Selección del perfil óptimo por grupo.** Evalúa cada perfil contra todas las barras del grupo y ordena por peso total.
9. **Lista desplegable de perfiles que cumplen.** Por grupo, con el detalle del perfil elegido.
10. **Desplazamientos por nodo y límite H/66.** Desplazamiento de cada combinación sísmica frente a H/66, con H ingresado por ti.

## 1. Configuración general

Materiales, factor de utilización por defecto y rutas. Edita aquí Fy, Ry y E.

In [ ]:
import math, re, glob, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Materiales (ingresar) ----
FY_MPA = 248.2      # Fy [MPa]. Modelo: A36 = 2531 kgf/cm2 (= 248.2 MPa)  <-- CONFIRMAR
RY = 1.5            # Ry (factor de fluencia esperada)                     <-- CONFIRMAR
E_MPA = 200000.0    # E [MPa]
USE_RY = True       # True: Ry·Fy en Tabla 9 (conservador)
# ---- Diseño ----
DESIGN = 'ASD'      # el .s2k del modelo está en ASD (AISC 360-16)
UTIL_MAX = 1.0      # factor de utilización máximo por defecto (se puede cambiar por grupo)
LIM_KLR_DEFAULT = 200.0
# ---- Rutas ----
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'
MODEL_FILE = None           # None = detectar .s2k o .txt con tablas SAP
BASE_DB = None              # None = buscar base_datos_perfiles.csv
# ---- Parámetros por grupo (editables). Una fila por grupo del modelo ----
# ---- Grupos: se detectan automáticamente desde el modelo (GROUPS 2 - ASSIGNMENTS) ----
# Cualquier grupo que crees en SAP aparece aquí. Para fijar valores de un grupo, edita GRUPOS_MANUAL.
# Lo que no indiques toma el valor por defecto según el nombre del grupo.
# Si subes parametros_grupos.csv a la entrada, sus filas tienen prioridad sobre GRUPOS_MANUAL.
GRUPOS_MANUAL = {
    # 'NOMBRE DEL GRUPO': dict(tipo_miembro='Viga', Kx=1.0, Ky=1.0, Lb_factor=1.0, Cb=1.0, util_max=1.0, lim_KLr=None, excepcion_0_7R1='No'),
}
# Catálogo completo para todos los grupos: IN, HN, W, IPE, IPN, HEA, HEB, tubos y ángulos.
# UPN y costaneras se listan en la base pero aún no tienen verificación implementada.
# Para restringir un grupo, cambia 'familias' en el formulario de la sección 3.
TODAS_LAS_FAMILIAS = 'I soldada,I/H ala ancha,I,H,I inclinada,HSS rectangular,HSS cuadrado,HSS circular,L (ángulo),C (canal)'
FAMILIAS_POR_TIPO = {'Columna': TODAS_LAS_FAMILIAS, 'Viga': TODAS_LAS_FAMILIAS, 'Arriostramiento': TODAS_LAS_FAMILIAS}
DEFAULT_GRUPO = dict(Kx=1.0, Ky=1.0, Lb_factor=1.0, Cb=1.0, util_max=UTIL_MAX, lim_KLr=None, excepcion_0_7R1='No')
# lim_KLr = None significa 1,5π√(E/Fy) (NCh2369 8.8.4), que se aplica a todos los grupos.
def tipo_por_nombre(nombre):
    n = nombre.upper()
    if 'COL' in n: return 'Columna'
    if 'VIG' in n: return 'Viga'
    if 'ARR' in n or 'DIAG' in n or 'PUNT' in n: return 'Arriostramiento'
    return 'Columna'   # por defecto: revisar en GRUPOS_MANUAL
_csv_grupos = glob.glob(f'{INPUT_DIR}/**/parametros_grupos.csv', recursive=True)
CSV_GRUPOS = pd.read_csv(_csv_grupos[0]) if _csv_grupos else None

def build_grupos(nombres):
    filas, sin_tipo = [], []
    for nombre in nombres:
        tipo = tipo_por_nombre(nombre)
        if tipo == 'Columna' and not any(k in nombre.upper() for k in ('COL',)):
            sin_tipo.append(nombre)
        d = dict(grupo=nombre, tipo_miembro=tipo, familias=FAMILIAS_POR_TIPO[tipo], **DEFAULT_GRUPO)
        d.update(GRUPOS_MANUAL.get(nombre, {}))
        if CSV_GRUPOS is not None and nombre in set(CSV_GRUPOS['grupo']):
            row = CSV_GRUPOS[CSV_GRUPOS['grupo'] == nombre].iloc[0].to_dict()
            d.update({k: v for k, v in row.items() if k != 'grupo' and pd.notna(v)})
        filas.append(d)
    if sin_tipo:
        print('Grupos sin tipo reconocido por nombre (se tomaron como Columna). Revísalos en GRUPOS_MANUAL:', sin_tipo)
    return pd.DataFrame(filas)
print('Fy =', FY_MPA, 'MPa | Ry =', RY, '| RyFy =', round(RY * FY_MPA, 1), 'MPa | E =', E_MPA, 'MPa')

## 2. Lectura del modelo SAP2000

Lee el archivo subido a `/kaggle/input`. No requiere edición.

In [ ]:
KV = re.compile(r'(\w+)=("[^"]*"|\S+)')
def find_model():
    if MODEL_FILE: return MODEL_FILE
    hits = [p for p in glob.glob(f'{INPUT_DIR}/**/*', recursive=True) if p.lower().endswith(('.s2k', '.txt'))]
    hits = [p for p in hits if open(p, encoding='latin-1', errors='replace').read(4000).find('TABLE:') >= 0]
    if len(hits) != 1:
        raise FileNotFoundError(f'Se esperaba un modelo .s2k/.txt en {INPUT_DIR}; encontrados: {hits}. Fija MODEL_FILE.')
    return hits[0]

def logical_lines(path):
    """Une las líneas que SAP parte con ' _' al final."""
    buf = ''
    with open(path, encoding='latin-1', errors='replace') as f:
        for line in f:
            s = line.rstrip('\n')
            if s.rstrip().endswith('_'):
                buf += s.rstrip()[:-1] + ' '
                continue
            yield buf + s
            buf = ''

WANT = {'JOINT COORDINATES', 'JOINT DISPLACEMENTS', 'PROGRAM CONTROL', 'PREFERENCES - STEEL DESIGN - AISC 360-16', 'CONNECTIVITY - FRAME',
        'FRAME SECTION ASSIGNMENTS', 'GROUPS 2 - ASSIGNMENTS', 'ELEMENT FORCES - FRAMES'}
def read_model(path, grupos):
    out = {k: [] for k in WANT}
    cur = None
    gset = None if grupos is None else set(grupos)
    frames_in_groups = set()
    for ln in logical_lines(path):
        s = ln.strip()
        if s.startswith('TABLE:'):
            m = re.match(r'TABLE:\s+"(.+?)"', s)
            cur = m.group(1) if m else None
            continue
        if cur not in WANT or '=' not in s:
            continue
        if cur == 'GROUPS 2 - ASSIGNMENTS':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if d.get('ObjectType') == 'Frame' and (gset is None or d.get('GroupName') in gset):
                out[cur].append((d['GroupName'], d['ObjectLabel']))
        elif cur == 'ELEMENT FORCES - FRAMES':
            if 'OutputCase="ASD' not in s and 'OutputCase=ASD' not in s:
                continue
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Frame'], float(d['Station']), d['OutputCase'], float(d['P']), float(d['V2']),
                             float(d['V3']), float(d['M2']), float(d['M3'])))
        elif cur == 'JOINT COORDINATES':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Joint'], float(d['GlobalX']), float(d['GlobalY']), float(d['GlobalZ'])))
        elif cur == 'JOINT DISPLACEMENTS':
            if 'OutputCase="ASD' not in s and 'OutputCase=ASD' not in s:
                continue
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Joint'], d['OutputCase'], d.get('StepType', 'Única'), float(d['U1']), float(d['U2']), float(d['U3'])))
        elif cur == 'PREFERENCES - STEEL DESIGN - AISC 360-16':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if 'OmegaB' in d: out[cur].append(d)
        elif cur == 'PROGRAM CONTROL':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if 'CurrUnits' in d: out[cur].append(d)
        else:
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append(d)
    return out

model_path = find_model()
print('Modelo:', model_path, '|', round(os.path.getsize(model_path) / 1e6, 1), 'MB')
raw = read_model(model_path, None)   # todos los grupos del modelo
units = raw['PROGRAM CONTROL'][0]['CurrUnits'] if raw['PROGRAM CONTROL'] else '?'
print('Unidades del modelo:', units)
assert [p.strip().lower() for p in units.split(',')][:2] == ['kgf', 'm'], 'El modelo debe estar en kgf-m'
pref = raw['PREFERENCES - STEEL DESIGN - AISC 360-16'][0]
OMEGA_C = float(pref.get('OmegaC', 1.67)); OMEGA_B = float(pref.get('OmegaB', 1.67))
OMEGA_T = float(pref.get('OmegaTY', 1.67)); OMEGA_V = float(pref.get('OmegaV', 1.67))
OMEGA_V_ROLLED = float(pref.get('OmegaVRolledI', 1.5))
print(f'Ω del modelo: Ωc={OMEGA_C} Ωb={OMEGA_B} Ωt={OMEGA_T} Ωv={OMEGA_V} Ωv(I laminado)={OMEGA_V_ROLLED}')

## 3. Ingreso de parámetros por grupo

Formulario para cada grupo del modelo. Los cambios se guardan automáticamente. Después de modificar, vuelve a ejecutar las celdas 8, 9 y 10.

In [ ]:
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

TIPOS = ['Columna', 'Viga', 'Arriostramiento']
LIM_8_8_4 = 1.5 * math.pi * math.sqrt(E_MPA / FY_MPA)
nombres_grupos = sorted({g for g, _ in raw['GROUPS 2 - ASSIGNMENTS']})
GRUPOS = build_grupos(nombres_grupos)
GRUPOS.to_csv(os.path.join(WORK_DIR, 'parametros_grupos_plantilla.csv'), index=False) if os.path.isdir(WORK_DIR) else None

def tabla_resumen():
    t = GRUPOS.copy()
    t['lim_KLr'] = t['lim_KLr'].apply(lambda v: f"1,5π√(E/Fy) = {LIM_8_8_4:.1f}" if (v is None or pd.isna(v)) else v)
    return t[['grupo', 'tipo_miembro', 'familias', 'Kx', 'Ky', 'Lb_factor', 'Cb', 'util_max', 'lim_KLr', 'excepcion_0_7R1']]

def _rebuild():
    global GRUPOS
    GRUPOS = build_grupos(nombres_grupos)

if not HAS_WIDGETS:
    print('ipywidgets no disponible: edita GRUPOS_MANUAL en la celda 1.')
    display(tabla_resumen())
else:
    _cargando = {'v': False}
    w_grupo = widgets.Dropdown(options=nombres_grupos, description='Grupo:', layout=widgets.Layout(width='420px'))
    w_tipo = widgets.Dropdown(options=TIPOS, description='Tipo:')
    w_fam = widgets.Text(description='Familias:', layout=widgets.Layout(width='600px'))
    w_Kx = widgets.FloatText(description='Kx:')
    w_Ky = widgets.FloatText(description='Ky:')
    w_Lb = widgets.FloatText(description='Lb factor:')
    w_Cb = widgets.FloatText(description='Cb:')
    w_util = widgets.FloatText(description='UC máx:')
    w_lim_auto = widgets.Checkbox(value=True, description='Límite KL/r = 1,5π√(E/Fy) (8.8.4)')
    w_lim = widgets.FloatText(description='KL/r máx:', disabled=True)
    w_exc = widgets.Dropdown(options=['No', 'Sí'], description='Excepción 8.8.4 (0,7R1≥1,0):')
    w_resumen = widgets.Output()

    def _cargar(nombre):
        _cargando['v'] = True
        f = GRUPOS[GRUPOS['grupo'] == nombre].iloc[0]
        w_tipo.value = f['tipo_miembro']; w_fam.value = f['familias']
        w_Kx.value = float(f['Kx']); w_Ky.value = float(f['Ky']); w_Lb.value = float(f['Lb_factor'])
        w_Cb.value = float(f['Cb']); w_util.value = float(f['util_max'])
        auto = f['lim_KLr'] is None or pd.isna(f['lim_KLr'])
        w_lim_auto.value = auto; w_lim.disabled = auto
        w_lim.value = 200.0 if auto else float(f['lim_KLr'])
        w_exc.value = f['excepcion_0_7R1']
        _cargando['v'] = False

    def _mostrar():
        with w_resumen:
            w_resumen.clear_output()
            display(tabla_resumen())

    def _guardar(*_):
        if _cargando['v']:
            return
        w_lim.disabled = w_lim_auto.value
        GRUPOS_MANUAL[w_grupo.value] = dict(
            tipo_miembro=w_tipo.value, familias=w_fam.value, Kx=w_Kx.value, Ky=w_Ky.value,
            Lb_factor=w_Lb.value, Cb=w_Cb.value, util_max=w_util.value,
            lim_KLr=None if w_lim_auto.value else float(w_lim.value), excepcion_0_7R1=w_exc.value)
        _rebuild()
        _mostrar()

    def _cambio_grupo(change):
        if change['name'] == 'value':
            _cargar(change['new'])
            _mostrar()

    w_grupo.observe(_cambio_grupo, names='value')
    for w in (w_tipo, w_fam, w_Kx, w_Ky, w_Lb, w_Cb, w_util, w_lim_auto, w_lim, w_exc):
        w.observe(_guardar, names='value')

    _cargar(nombres_grupos[0])
    display(HTML('<b>Ingreso de parámetros por grupo.</b> Cambia un valor y se guarda solo. '
                 'Al terminar, ejecuta las celdas 8, 9 y 10 para recalcular.<br>'
                 'Tipos: Columna, Viga o Arriostramiento. Familias: separadas por coma '
                 '(I soldada, I/H ala ancha, I, H, HSS cuadrado, HSS rectangular, HSS circular, L (ángulo)).'))
    display(widgets.VBox([w_grupo,
                          widgets.HBox([w_tipo, w_Kx, w_Ky, w_Lb, w_Cb, w_util]),
                          w_fam,
                          widgets.HBox([w_lim_auto, w_lim]),
                          w_exc,
                          w_resumen]))
    _mostrar()

## 4. Barras, grupos y demandas ASD

Une barras con grupos y arma las demandas por combinación y estación. No requiere edición.

In [ ]:
KGF = 9.80665
conn = pd.DataFrame(raw['CONNECTIVITY - FRAME'])[['Frame', 'Length']]
conn['L_mm'] = conn['Length'].astype(float) * 1000
sap_sec = pd.DataFrame(raw['FRAME SECTION ASSIGNMENTS'])[['Frame', 'AnalSect']]
fr = pd.DataFrame(raw['GROUPS 2 - ASSIGNMENTS'], columns=['grupo', 'Frame'])
fr = fr.merge(conn, on='Frame', how='left').merge(sap_sec, on='Frame', how='left')
fr_base = fr.copy()   # parámetros de grupo se unen en la celda de selección
missing = fr[fr['L_mm'].isna()]
assert missing.empty, f'Barras de grupo sin conectividad: {missing.Frame.tolist()}'
print('Barras en grupos:', len(fr), '| por grupo:', fr.groupby('grupo').size().to_dict())

dem = pd.DataFrame(raw['ELEMENT FORCES - FRAMES'], columns=['Frame', 'Station', 'combo', 'P', 'V2', 'V3', 'M2', 'M3'])
dem = dem[dem['Frame'].isin(fr['Frame'])].copy()
dem['P'] *= KGF; dem['V2'] *= KGF; dem['V3'] *= KGF
dem['M2'] *= KGF * 1000; dem['M3'] *= KGF * 1000          # kgf·m -> N·mm
dem = dem.merge(fr[['Frame', 'grupo']], on='Frame')
print('Filas de demanda ASD:', len(dem), '| combinaciones:', dem['combo'].nunique())
print(dem.groupby('grupo').agg(Pc_max_kN=('P', lambda s: round(-s.min()/1000, 1)),
                              Pt_max_kN=('P', lambda s: round(s.max()/1000, 1)),
                              M3_max_kNm=('M3', lambda s: round(s.abs().max()/1e6, 2)),
                              M2_max_kNm=('M2', lambda s: round(s.abs().max()/1e6, 2))))

## 5. Base de datos de perfiles (oculta)

Base de datos incrustada, codificada en una sola cadena para que no ocupe espacio. Está oculta por defecto. No requiere edición.

In [ ]:
# Base de datos de perfiles (codificada). No requiere edición.
import gzip, base64, io
DB_GZ_B64 = (
    "..."
)
def find_db():
    if BASE_DB: return BASE_DB
    hits = glob.glob(f"{INPUT_DIR}/**/base_datos_perfiles.csv", recursive=True)
    return hits[0] if len(hits) == 1 else None
_db_path = find_db()
if _db_path:
    db_raw = pd.read_csv(_db_path)
else:
    db_raw = pd.read_csv(io.StringIO(gzip.decompress(base64.b64decode(DB_GZ_B64)).decode("utf-8")))
print("Base de datos de perfiles:", _db_path or "incrustada", "|", len(db_raw), "perfiles")

## 6. Catálogo de propiedades

Convierte la base de datos a unidades de cálculo (mm, N) y deriva las propiedades necesarias. No requiere edición.

In [ ]:
if 'db_raw' not in globals():
    raise RuntimeError('Falta la base de datos de perfiles. Ejecuta primero la celda 5 '
                       '("Base de datos de perfiles (oculta)") y luego vuelve a ejecutar esta celda.')
def fl(v):
    return float(v) if pd.notna(v) and v != '' else np.nan

def props(r):
    """Propiedades en mm, N. tipo_kind: I | RECT | ROUND | ANG | OTHER."""
    t = r['tipo']
    A = fl(r['A_cm2']) * 100
    kind = {'C (canal)': 'C', 'I soldada': 'I', 'I/H ala ancha': 'I', 'I': 'I', 'H': 'I', 'I inclinada': 'I',
            'HSS rectangular': 'RECT', 'HSS cuadrado': 'RECT', 'HSS circular': 'ROUND',
            'L (ángulo)': 'ANG'}.get(t, 'OTHER')
    d = dict(kind=kind, A=A, peso=fl(r['peso_kgf_m']))
    if kind == 'C':
        H, B, tw, tf = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm']), fl(r['tf_mm'])
        Ix, Iy = fl(r['Ix']) * 1e4, fl(r['Iy']) * 1e4
        Sx = fl(r['Wx']) * 1e3
        Sy = fl(r['Wy']) * 1e3
        d.update(H=H, B=B, tw=tw, tf=tf, Ix=Ix, Iy=Iy, Sx=Sx, Sy=Sy, Zx=Sx, Zy=Sy,
                 J=fl(r['J']) * 1e4 if pd.notna(r['J']) else np.nan,
                 Cw=fl(r['Cw_cm6']) * 1e6 if pd.notna(r['Cw_cm6']) else np.nan)
        d['rx'] = np.sqrt(Ix / d['A']); d['ry'] = np.sqrt(Iy / d['A'])
    if kind == 'I':
        H, B, tw, tf = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm']), fl(r['tf_mm'])
        Zx = fl(r['Zx']) * 1e3 if pd.notna(r['Zx']) else (B * tf * (H - tf) + tw * (H - 2 * tf) ** 2 / 4)
        # Zy no viene en la planilla de vigas soldadas (HN): aproximación geométrica de la sección I
        Zy = fl(r['Zy']) * 1e3 if pd.notna(r['Zy']) else (2 * tf * B ** 2 / 4 + (H - 2 * tf) * tw ** 2 / 4)
        d.update(H=H, B=B, tw=tw, tf=tf,
                 Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4, Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3,
                 Zx=Zx, Zy=Zy, J=fl(r['J']) * 1e4 if pd.notna(r['J']) else np.nan)
        d['Cw'] = d['Iy'] * (H - tf) ** 2 / 4
    elif kind == 'RECT':
        H, B, t = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm'])
        d.update(H=H, B=B, tw=t, tf=t, Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4,
                 Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3)
        d['Zx'] = d['Sx'] * 1.12; d['Zy'] = d['Sy'] * 1.12        # aprox. para tubo rectangular (no viene en la planilla)
        Am = (B - t) * (H - t); p = 2 * ((B - t) + (H - t))
        d['J'] = 4 * Am ** 2 * t / p                               # torsión de sección cerrada de pared delgada
        d['Cw'] = 0.0
    elif kind == 'ROUND':
        D, t = fl(r['H_mm']), fl(r['tw_mm'])
        I = fl(r['Ix']) * 1e4
        d.update(H=D, B=D, tw=t, tf=t, Ix=I, Iy=I, Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wx']) * 1e3,
                 Zx=fl(r['Wx']) * 1e3 * 4 / 3 if pd.notna(r['Wx']) else np.nan)
        d['Zy'] = d['Zx']; d['J'] = 2 * I; d['Cw'] = 0.0
    elif kind == 'ANG':
        leg, t = fl(r['H_mm']), fl(r['tw_mm'])
        d.update(H=leg, B=fl(r['B_mm']), tw=t, tf=t, Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4,
                 Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3)
        d['rx'] = fl(r['ix']) * 10
        d['ry'] = fl(r['r_min_cm']) * 10 if pd.notna(r['r_min_cm']) and r['r_min_cm'] != '' else np.nan
        # Z plástico no viene en la planilla de ángulos: se aproxima como 1,5·S (flexión de ángulos no es el control)
        d['Zx'] = 1.5 * d['Sx']; d['Zy'] = 1.5 * d['Sy']
    return d

cat = db_raw[db_raw['tipo'].isin(['C (canal)', 'I soldada', 'I/H ala ancha', 'I', 'H', 'I inclinada', 'HSS rectangular',
                                   'HSS cuadrado', 'HSS circular', 'L (ángulo)'])].copy()
cat = cat[cat['validado'].astype(str).isin(['sí', 'si', 'True', 'true'])]
cat = cat[cat['A_cm2'].notna() & cat['Ix'].notna()]
PROPS = []
for _, r in cat.iterrows():
    d = props(r)
    if d['kind'] != 'ANG':
        d['rx'] = np.sqrt(d['Ix'] / d['A'])
        d['ry'] = np.sqrt(d['Iy'] / d['A'])
    d.update(designacion=r['designacion'], tipo=r['tipo'], familia=r['familia'],
             laminado=str(r['familia']).startswith('Copromet') or str(r['familia']).startswith('Cintac'))
    PROPS.append(d)
PROPS = [p for p in PROPS if not (p['kind'] == 'ANG' and not np.isfinite(p['ry']))]
cand = pd.DataFrame(PROPS)
print('Perfiles candidatos (validados, tipos soportados):', len(cand))
print(cand.groupby('tipo').size().to_dict())

## 7. Verificación ASD y límites de esbeltez

Fórmulas de AISC 360-16 ASD y límites de la NCh2369:2025 (Tabla 9 y 8.8.4). No requiere edición.

In [ ]:
E = E_MPA
FYE = FY_EFF = RY * FY_MPA if USE_RY else FY_MPA
LAM_ALAS_NO_ATIESADA = 0.40 * math.sqrt(E / FYE)     # Tabla 9, alas no atiesadas I/H (2025)
LAM_ALMA_ARR = 1.57 * math.sqrt(E / FYE)              # Tabla 9, almas I/H/C arriostramiento (2025)
LAM_ALMA_FLEX_A = lambda ca: 3.96 * math.sqrt(E / FYE) * (1 - 3.04 * ca)  # Tabla 9, almas flexo-compresión, Ca<=0.114 (2025)
LAM_HSS_WALL = 1.40 * math.sqrt(E / FYE)              # 2003 Tabla 8.1 rectangular espesor uniforme (completa 2025)
LAM_ANG = 0.45 * math.sqrt(E / FYE)                   # 2003 Tabla 8.1 puntales de ángulos (completa)
LAM_GLOBAL_ARR = 1.5 * math.pi * math.sqrt(E / FY_MPA)  # NCh2369:2025 8.8.4 (usa Fy, no Ry·Fy)

def fcr_comp(lam, A):
    """Pn = Fcr·A según AISC E3 (lam = KL/r máx)."""
    Fe = math.pi ** 2 * E / lam ** 2
    if lam <= 4.71 * math.sqrt(E / FY_MPA):
        Fcr = 0.658 ** (FY_MPA / Fe) * FY_MPA
    else:
        Fcr = 0.877 * Fe
    return Fcr * A

def Mn_major(c, Lb, Cb):
    """Momento nominal eje fuerte (F2 para I doblemente simétricas; Mp en tubos y ángulos)."""
    Mp = FY_MPA * c['Zx']
    if c['kind'] == 'C':
        return min(Mp, 1.5 * FY_MPA * c['Sx'])   # canal: sin pandeo lateral-torsional (pendiente); conservador
    if c['kind'] != 'I' or not np.isfinite(c['J']):
        return Mp
    ry = c['ry']; Sx = c['Sx']; ho = c['H'] - c['tf']; J = c['J']
    Lp = 1.76 * ry * math.sqrt(E / FY_MPA)
    rts = math.sqrt(math.sqrt(c['Iy'] * c['Cw']) / Sx)
    Lr = 1.95 * rts * E / (0.7 * FY_MPA) * math.sqrt(J / (Sx * ho)) * \
         math.sqrt(1 + math.sqrt(1 + 6.76 * (0.7 * FY_MPA * Sx * ho / (E * J)) ** 2))
    if Lb <= Lp:
        return Mp
    if Lb <= Lr:
        return min(Cb * (Mp - (Mp - 0.7 * FY_MPA * Sx) * (Lb - Lp) / (Lr - Lp)), Mp)
    Fcr = Cb * math.pi ** 2 * E / (Lb / rts) ** 2 * math.sqrt(1 + 0.078 * J / (Sx * ho) * (Lb / rts) ** 2)
    return min(Fcr * Sx, Mp)

def Mn_minor(c):
    return min(FY_MPA * c['Zy'], 1.6 * FY_MPA * c['Sy'])

def Vn(c, axis):
    if c['kind'] == 'C':
        Aw = c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tf']
        return 0.6 * FY_MPA * Aw
    if c['kind'] == 'I':
        Aw = c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tf']
    elif c['kind'] == 'RECT':
        Aw = 2 * c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tw']
    elif c['kind'] == 'ROUND':
        Aw = 0.5 * c['A']
    else:
        return np.nan
    return 0.6 * FY_MPA * Aw

def check_section(c, frames_df, dem_g):
    """Devuelve (UC máximo, detalle) para un perfil frente a todas las barras de un grupo."""
    tipo = frames_df['tipo_miembro'].iloc[0]
    if c['kind'] == 'OTHER':
        return np.inf, {'motivo': 'tipo no soportado'}
    if c['kind'] == 'ANG' and not np.isfinite(c['ry']):
        return np.inf, {'motivo': 'sin r_min'}
    Ca_max = 0.0
    if c['kind'] in ('I', 'C') and tipo != 'Arriostramiento':
        Ca_max = max(0.0, OMEGA_C * (-dem_g['P'].min()) / (FYE * c['A']))
    uc_all, g_all = 0.0, {}
    for _, fr_ in frames_df.iterrows():
        dfr = dem_g[dem_g['Frame'] == fr_['Frame']]
        if dfr.empty:
            continue
        L = fr_['L_mm']
        kx, ky = fr_['Kx'] * L, fr_['Ky'] * L
        rx = c['rx']; ry = c['ry']
        lam = max(kx / rx, ky / ry)
        Pn = fcr_comp(lam, c['A'])
        Pc = Pn / OMEGA_C
        Tc = FY_MPA * c['A'] / OMEGA_T
        Mn3 = Mn_major(c, L * fr_['Lb_factor'], fr_['Cb'])
        Mn2 = Mn_minor(c)
        M3c, M2c = Mn3 / OMEGA_B, Mn2 / OMEGA_B
        V3c, V2c = Vn(c, 3), Vn(c, 2)
        omv3 = OMEGA_V_ROLLED if (c['laminado'] and c['kind'] == 'I' and c['tipo'] != 'I soldada') else OMEGA_V
        V3c = V3c / omv3 if np.isfinite(V3c) else np.inf
        V2c = V2c / OMEGA_V if np.isfinite(V2c) else np.inf
        P = dfr['P'].values
        comp = np.maximum(-P, 0)
        tens = np.maximum(P, 0)
        ratio_P = np.where(P < 0, comp / Pc, tens / Tc)
        cap = np.where(P < 0, Pc, Tc)
        m3 = np.abs(dfr['M3'].values) / M3c
        m2 = np.abs(dfr['M2'].values) / M2c
        ratio = np.where(ratio_P >= 0.2, ratio_P + 8 / 9 * (m3 + m2), ratio_P / 2 + (m3 + m2))
        vr3 = np.abs(dfr['V3'].values) / V3c if np.isfinite(V3c) else np.zeros(len(P))
        vr2 = np.abs(dfr['V2'].values) / V2c if np.isfinite(V2c) else np.zeros(len(P))
        ratio = np.maximum(ratio, np.maximum(vr3, vr2))
        if not np.all(np.isfinite(ratio)):
            return np.inf, {'motivo': 'resultado no finito'}
        uc = float(ratio.max())
        if uc > uc_all:
            uc_all = uc
            i = int(np.argmax(ratio))
            g_all = dict(frame=fr_['Frame'], combo=dfr['combo'].iloc[i], lam=round(lam, 1), UC=round(uc, 3))
        g_all.setdefault('lam_max', 0.0)
        g_all['lam_max'] = max(g_all['lam_max'], round(lam, 1))
    return uc_all, g_all

def check_local(c, tipo, Ca_max):
    """Relaciones ancho/espesor frente a los límites de NCh2369. Devuelve (ok, detalle)."""
    sq = math.sqrt(E / FYE)
    if c['kind'] in ('I', 'C'):
        lam_f = ((c['B'] / 2) if c['kind'] == 'I' else c['B']) / c['tf']
        lim_f = LAM_ALAS_NO_ATIESADA
        h_clear = c['H'] - 2 * c['tf']
        lam_w = h_clear / c['tw']
        if tipo == 'Arriostramiento':
            lim_w = LAM_ALMA_ARR
        else:
            lim_w = LAM_ALMA_FLEX_A(Ca_max) if Ca_max <= 0.114 else 1.57 * sq     # Ca>0.114: piso conservador
        ok = lam_f <= lim_f and lam_w <= lim_w
        return ok, dict(lam_ala=round(lam_f, 2), lim_ala=round(lim_f, 2), lam_alma=round(lam_w, 2), lim_alma=round(lim_w, 2))
    if c['kind'] == 'RECT':
        lam_b = (c['B'] - 2 * c['tw']) / c['tw']; lam_h = (c['H'] - 2 * c['tw']) / c['tw']
        ok = lam_b <= LAM_HSS_WALL and lam_h <= LAM_HSS_WALL
        return ok, dict(lam_ala=round(lam_b, 2), lim_ala=round(LAM_HSS_WALL, 2), lam_alma=round(lam_h, 2), lim_alma=round(LAM_HSS_WALL, 2))
    if c['kind'] == 'ROUND':
        lam = c['H'] / c['tw']
        lim = (0.062 if tipo == 'Arriostramiento' else 0.077) * E / FYE
        return lam <= lim, dict(lam_ala=round(lam, 2), lim_ala=round(lim, 2), lam_alma=None, lim_alma=None)
    if c['kind'] == 'ANG':
        lam = c['H'] / c['tw']
        return lam <= LAM_ANG, dict(lam_ala=round(lam, 2), lim_ala=round(LAM_ANG, 2), lam_alma=None, lim_alma=None)
    return False, {}

## 8. Selección del perfil óptimo por grupo

Evalúa cada perfil contra todas las barras del grupo. Ejecuta esta celda después de cambiar parámetros.

In [ ]:
# Parámetros actuales por grupo (vienen del panel de la celda 3)
fr = fr_base.merge(GRUPOS[['grupo', 'tipo_miembro', 'Kx', 'Ky', 'Lb_factor', 'Cb', 'util_max', 'lim_KLr', 'excepcion_0_7R1']], on='grupo')
RESULT_COLS = ['grupo', 'tipo_miembro', 'designacion', 'familia', 'tipo', 'peso_kgf_m', 'peso_total_kgf',
               'UC_max', 'util_max', 'lam_max', 'lim_lam', 'local_ok', 'global_ok', 'cumple', 'barra_critica',
               'combo_critica', 'detalle_local', 'sap_actual']
def run_group(g):
    """g: fila de GRUPOS. Evalúa todos los candidatos de sus familias y ordena por peso total."""
    fam_ok = [s.strip() for s in g['familias'].split(',')]
    frames_g = fr[fr['grupo'] == g['grupo']]
    dem_g = dem[dem['grupo'] == g['grupo']]
    c_pool = cand[cand['tipo'].isin(fam_ok)]
    lim_val = g.get('lim_KLr')
    lim_g = LAM_GLOBAL_ARR if (lim_val is None or pd.isna(lim_val)) else float(lim_val)
    rows = []
    sap_list = ', '.join(sorted(frames_g['AnalSect'].dropna().unique()))
    for _, c in c_pool.iterrows():
        c = c.to_dict()
        uc, det = check_section(c, frames_g, dem_g)
        if not np.isfinite(uc):
            continue
        Ca_max = 0.0
        if c['kind'] in ('I', 'C') and g['tipo_miembro'] != 'Arriostramiento':
            Ca_max = max(0.0, OMEGA_C * (-dem_g['P'].min()) / (FYE * c['A']))
        ok_loc, det_loc = check_local(c, g['tipo_miembro'], Ca_max)
        lam_max = det.get('lam_max', 0.0)
        ok_glob = lam_max <= lim_g
        if str(g.get('excepcion_0_7R1', 'No')).strip().lower() in ('sí', 'si', 'yes', 'true'):
            ok_loc, ok_glob = True, True          # 8.8.4: exceptuado (0,7·R1 ≥ 1,0); el usuario debe confirmarlo
            det_loc = {'nota': 'excepción 8.8.4 (0,7R1≥1,0) aplicada'}
        peso_tot = float((frames_g['L_mm'] / 1000 * c['peso']).sum()) if np.isfinite(c['peso']) else np.nan
        cumple = (uc <= g['util_max']) and ok_loc and ok_glob
        rows.append(dict(grupo=g['grupo'], tipo_miembro=g['tipo_miembro'], designacion=c['designacion'],
                         familia=c['familia'], tipo=c['tipo'], peso_kgf_m=round(c['peso'], 2),
                         peso_total_kgf=round(peso_tot, 1), UC_max=round(uc, 3), util_max=g['util_max'],
                         lam_max=lam_max, lim_lam=round(lim_g, 1), local_ok=ok_loc, global_ok=ok_glob,
                         cumple=cumple, barra_critica=det.get('frame'), combo_critica=det.get('combo'),
                         detalle_local=str(det_loc), sap_actual=sap_list))
    res = pd.DataFrame(rows, columns=RESULT_COLS)
    return res.sort_values(['cumple', 'peso_total_kgf'], ascending=[False, True])

RESULTADOS = {}
for _, g in GRUPOS.iterrows():
    RESULTADOS[g['grupo']] = run_group(g)
    r = RESULTADOS[g['grupo']]
    ok = r[r['cumple']]
    print(f"\n=== {g['grupo']} ({g['tipo_miembro']}) | candidatos evaluados: {len(r)} | cumplen: {len(ok)} | util_max={g['util_max']}")
    print('    SAP actual:', r['sap_actual'].iloc[0] if len(r) else '-')
    if len(ok):
        print(ok.head(8)[['designacion', 'peso_kgf_m', 'peso_total_kgf', 'UC_max', 'lam_max', 'lim_lam', 'local_ok', 'global_ok', 'barra_critica', 'combo_critica']].to_string(index=False))
    else:
        print('    Ningún perfil del catálogo cumple. Motivo más frecuente (UC > util_max o esbeltez):')
        print(r[['designacion', 'UC_max', 'local_ok', 'global_ok']].head(5).to_string(index=False))

os.makedirs(WORK_DIR, exist_ok=True)
pd.concat(RESULTADOS.values()).to_csv(os.path.join(WORK_DIR, 'resultados_optimos_por_grupo.csv'), index=False)
print('\nResultados guardados en', os.path.join(WORK_DIR, 'resultados_optimos_por_grupo.csv'))

## 9. Lista desplegable de perfiles que cumplen

Por grupo, lista los perfiles que cumplen. Selecciona uno para ver su detalle.

In [ ]:
# %% [interactivo] LISTA DESPLEGABLE DE PERFILES QUE CUMPLEN, POR GRUPO
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

def resumen_perfil(row):
    txt = [f"<b>{row['designacion']}</b> ({row['tipo']}) — peso {row['peso_kgf_m']} kgf/m, total grupo {row['peso_total_kgf']} kgf",
           f"UC máx = {row['UC_max']} (límite {row['util_max']}) · barra crítica {row['barra_critica']} · combo {row['combo_critica']}",
           f"Esbeltez global máx = {row['lam_max']} (límite {row['lim_lam']}) · local: {row['detalle_local']}"]
    return '<br>'.join(txt)

def tabla_cumplen(res):
    ok = res[res['cumple']]
    return ok[['designacion', 'tipo', 'peso_kgf_m', 'peso_total_kgf', 'UC_max', 'lam_max', 'lim_lam']]

PANELES = {}
for grupo, res in RESULTADOS.items():
    ok = res[res['cumple']].reset_index(drop=True)
    if ok.empty:
        display(HTML(f'<h4>{grupo}</h4><p>Ningún perfil del catálogo cumple con los parámetros actuales.</p>'))
        continue
    if HAS_WIDGETS:
        opciones = [(f"{r.designacion}  |  {r.peso_total_kgf} kgf  |  UC {r.UC_max}", i) for i, r in ok.iterrows()]
        dd = widgets.Dropdown(options=opciones, description='Perfil:', layout=widgets.Layout(width='700px'))
        salida = widgets.Output()
        def _mostrar(change, ok=ok, salida=salida):
            with salida:
                salida.clear_output()
                display(HTML(resumen_perfil(ok.loc[change['new']])))
        dd.observe(_mostrar, names='value')
        with salida:
            display(HTML(resumen_perfil(ok.loc[0])))
        PANELES[grupo] = widgets.VBox([widgets.HTML(f'<h4>{grupo}: {len(ok)} perfiles cumplen (mejor primero)</h4>'), dd, salida])
        display(PANELES[grupo])
    else:
        display(HTML(f'<h4>{grupo}: {len(ok)} perfiles cumplen</h4>'))
        display(tabla_cumplen(res))

## 10. Desplazamientos por nodo y límite H/66

Elige nodo y referencia, ingresa H y revisa los desplazamientos por combinación sísmica.

In [ ]:
# %% [desplazamiento] REVISIÓN DE DESPLAZAMIENTOS POR NODO Y COMBINACIÓN SÍSMICA
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

# ---- DATOS EDITABLES ----
DIVISOR_DRIFT = 66          # límite admisible H/66 (NCh2369)
NODO_SEL = None             # None = nodo de mayor cota Z (techo). O código de nodo, p. ej. '52'
NODO_REF = None             # None = nodo de la base más cercano en planta. O código de nodo
H_M = 2.84                  # altura de piso H [m] con la que se calcula H/66  <-- editable

# ---- DATOS ----
nod = pd.DataFrame(raw['JOINT COORDINATES'], columns=['Joint', 'X', 'Y', 'Z']).sort_values(['Z', 'X', 'Joint']).reset_index(drop=True)
dsp = pd.DataFrame(raw['JOINT DISPLACEMENTS'], columns=['Joint', 'combo', 'paso', 'U1', 'U2', 'U3'])
seis = sorted(c for c in dsp['combo'].unique() if re.search(r'E[xyz]', c))
dsp = dsp[dsp['combo'].isin(seis)]
nod_idx = nod.set_index('Joint')
base_nodos = nod[nod['Z'].abs() < 0.05]

def nodo_base_cercano(j):
    p = nod_idx.loc[j]
    b = base_nodos.assign(d=np.hypot(base_nodos['X'] - p['X'], base_nodos['Y'] - p['Y'])).sort_values('d')
    return b.iloc[0]['Joint'] if len(b) else nod.iloc[0]['Joint']

def tabla_desplazamientos(nodo, ref, H):
    """Envolvente (Max/Min de SAP) del desplazamiento relativo del nodo respecto a la referencia, frente a H/66."""
    lim_mm = H * 1000 / DIVISOR_DRIFT
    rows = []
    for combo in seis:
        dc = dsp[dsp['combo'] == combo]
        n = dc[dc['Joint'] == nodo]; r = dc[dc['Joint'] == ref]
        n1, n2 = n['U1'].values * 1000, n['U2'].values * 1000
        r1, r2 = r['U1'].values * 1000, r['U2'].values * 1000
        # envolvente conservadora: nodo en un extremo y referencia en el otro
        dX = max(abs(n1.max() - r1.min()), abs(n1.min() - r1.max()))
        dY = max(abs(n2.max() - r2.min()), abs(n2.min() - r2.max()))
        rows.append({'combinación': combo,
                     'U1 máx (mm)': round(n1.max(), 2), 'U1 mín (mm)': round(n1.min(), 2),
                     'U2 máx (mm)': round(n2.max(), 2), 'U2 mín (mm)': round(n2.min(), 2),
                     'Δ X rel. (mm)': round(dX, 2), 'Δ Y rel. (mm)': round(dY, 2),
                     'deriva X': round(dX / (H * 1000), 6), 'deriva Y': round(dY / (H * 1000), 6),
                     'límite H/66 (mm)': round(lim_mm, 2),
                     'cumple': bool(max(dX, dY) <= lim_mm)})
    return pd.DataFrame(rows)

def resumen(nodo, ref, H):
    t = tabla_desplazamientos(nodo, ref, H)
    lim = H / DIVISOR_DRIFT
    txt = (f"<b>Nodo {nodo}</b> (Z = {nod_idx.loc[nodo, 'Z']:.2f} m) respecto a nodo <b>{ref}</b> (Z = {nod_idx.loc[ref, 'Z']:.2f} m) · "
           f"H = {H} m · límite H/{DIVISOR_DRIFT} = {lim*1000:.2f} mm · combinaciones que NO cumplen: {int((~t['cumple']).sum())} de {len(t)}")
    return HTML(txt), t

NODO0 = NODO_SEL or nod.iloc[-1]['Joint']
REF0 = NODO_REF or nodo_base_cercano(NODO0)
opciones_nodo = [(f"{r.Joint} · Z={r.Z:.2f} · X={r.X:.2f} · Y={r.Y:.2f}", r.Joint) for r in nod.itertuples()]
opciones_ref = [(f"{r.Joint} · Z={r.Z:.2f} · X={r.X:.2f} · Y={r.Y:.2f}", r.Joint) for r in base_nodos.itertuples()]

if HAS_WIDGETS:
    w_nodo = widgets.Dropdown(options=opciones_nodo, value=NODO0, description='Nodo:', layout=widgets.Layout(width='420px'))
    w_ref = widgets.Dropdown(options=opciones_ref, value=REF0, description='Referencia:', layout=widgets.Layout(width='420px'))
    w_H = widgets.FloatText(value=H_M, description='H (m):', layout=widgets.Layout(width='200px'))
    salida = widgets.Output()
    def _actualizar(_=None):
        with salida:
            salida.clear_output()
            cab, t = resumen(w_nodo.value, w_ref.value, w_H.value)
            display(cab)
            display(t)
    for w in (w_nodo, w_ref, w_H):
        w.observe(_actualizar, names='value')
    display(widgets.VBox([widgets.HBox([w_nodo, w_ref, w_H]), salida]))
    _actualizar()
else:
    cab, t = resumen(NODO0, REF0, H_M)
    display(cab); display(t)